# CSCE 40103 Module 2 Section 3: Security Data Visualization

This notebook demonstrates visual security EDA using PhiUSIIL phishing URL data and UNSW-NB15 network intrusion data. Each plot is tied to a security question.

## Data Setup

This notebook uses two cybersecurity datasets throughout the demonstrations. Place dataset CSV files in a folder named `data` next to this notebook. PhiUSIIL can be downloaded automatically from the UCI archive when internet access is available. UNSW-NB15 should be placed locally from the official dataset page.

In [ ]:
# Core libraries
from pathlib import Path
import io
import zipfile
import urllib.request
import warnings

import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False

# scikit-learn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay,
    average_precision_score, precision_recall_curve, roc_curve,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score
)
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.utils import resample

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 140)

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)

PHISHING_UCI_ZIP_URL = 'https://archive.ics.uci.edu/static/public/967/phiusiil+phishing+url+dataset.zip'
PHISHING_CSV_NAME = 'PhiUSIIL_Phishing_URL_Dataset.csv'

UNSW_LOCAL_NAMES = [
    'UNSW_NB15_training-set.csv',
    'UNSW_NB15_testing-set.csv',
    'UNSW-NB15_1.csv',
    'UNSW-NB15.csv'
]

UNSW_HELP = """
UNSW-NB15 was not found locally.
Place the official UNSW-NB15 CSV file in the data folder using one of these names:
- UNSW_NB15_training-set.csv
- UNSW_NB15_testing-set.csv
- UNSW-NB15_1.csv
- UNSW-NB15.csv
Official dataset page: https://research.unsw.edu.au/projects/unsw-nb15-dataset
"""

def _find_file(names):
    candidates = []
    for name in names:
        candidates.extend([DATA_DIR / name, Path(name)])
    for path in candidates:
        if path.exists():
            return path
    return None

def load_phiusiil(force_download=False):
    local = _find_file([PHISHING_CSV_NAME, 'phiusiil.csv', 'phishing.csv'])
    if local is not None and not force_download:
        df = pd.read_csv(local)
        print(f'Loaded PhiUSIIL from {local} with shape {df.shape}.')
        return df

    zip_path = DATA_DIR / 'phiusiil_phishing_url_dataset.zip'
    try:
        if force_download or not zip_path.exists():
            print('Downloading PhiUSIIL dataset from UCI...')
            urllib.request.urlretrieve(PHISHING_UCI_ZIP_URL, zip_path)
        with zipfile.ZipFile(zip_path) as z:
            csv_members = [m for m in z.namelist() if m.lower().endswith('.csv')]
            target = PHISHING_CSV_NAME if PHISHING_CSV_NAME in csv_members else csv_members[0]
            with z.open(target) as f:
                df = pd.read_csv(f)
        print(f'Loaded PhiUSIIL from UCI archive with shape {df.shape}.')
        return df
    except Exception as exc:
        raise FileNotFoundError(
            'PhiUSIIL dataset could not be loaded. Place PhiUSIIL_Phishing_URL_Dataset.csv in the data folder.'
        ) from exc

def load_unsw():
    local = _find_file(UNSW_LOCAL_NAMES)
    if local is None:
        print(UNSW_HELP)
        return None
    df = pd.read_csv(local)
    print(f'Loaded UNSW-NB15 from {local} with shape {df.shape}.')
    return df

def normalize_colnames(df):
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    return out

def get_label_col(df):
    for c in ['label', 'Label', 'CLASS_LABEL', 'class', 'target']:
        if c in df.columns:
            return c
    candidates = [c for c in df.columns if 'label' in str(c).lower() or 'class' in str(c).lower()]
    if candidates:
        return candidates[0]
    raise ValueError('No label column found. Inspect df.columns and set the label column manually.')

def binary_label_names(series, positive_name='Positive', negative_name='Negative'):
    values = pd.Series(series).dropna().unique().tolist()
    mapping = {}
    if set(values).issubset({0, 1, 0.0, 1.0}):
        mapping = {0: negative_name, 1: positive_name, 0.0: negative_name, 1.0: positive_name}
    return pd.Series(series).map(mapping).fillna(pd.Series(series).astype(str))

def prepare_datasets():
    phish = normalize_colnames(load_phiusiil())
    unsw = load_unsw()
    if unsw is not None:
        unsw = normalize_colnames(unsw)
    return phish, unsw

phish, unsw = prepare_datasets()
phish_label = get_label_col(phish)
unsw_label = get_label_col(unsw) if unsw is not None else None

print('PhiUSIIL label column:', phish_label)
if unsw is not None:
    print('UNSW-NB15 label column:', unsw_label)

In [ ]:
def pick_existing(df, candidates, minimum=1):
    cols = [c for c in candidates if c in df.columns]
    if len(cols) < minimum:
        raise ValueError(f'Could not find enough columns. Tried: {candidates}')
    return cols

def safe_numeric_frame(df, preferred=None, max_cols=12):
    if preferred is None:
        preferred = []
    cols = [c for c in preferred if c in df.columns]
    remaining = [c for c in df.select_dtypes(include='number').columns if c not in cols]
    cols = (cols + remaining)[:max_cols]
    return df[cols].copy(), cols

def add_target_names(phish_df, unsw_df=None):
    p = phish_df.copy()
    p['target_name'] = binary_label_names(p[phish_label], positive_name='Phishing', negative_name='Legitimate')
    u = None
    if unsw_df is not None and unsw_label is not None:
        u = unsw_df.copy()
        u['target_name'] = binary_label_names(u[unsw_label], positive_name='Attack', negative_name='Normal')
    return p, u

def sample_df(df, n=5000, random_state=7):
    if df is None:
        return None
    return df.sample(min(n, len(df)), random_state=random_state) if len(df) > n else df.copy()

def build_basic_xy(df, label_col, preferred_numeric=None, preferred_categorical=None, max_numeric=12, max_categorical=8, drop_cols=None):
    drop_cols = set(drop_cols or []) | {label_col}
    if 'attack_cat' in df.columns and label_col == 'label':
        drop_cols.add('attack_cat')
    numeric_candidates = [c for c in (preferred_numeric or []) if c in df.columns and c not in drop_cols]
    categorical_candidates = [c for c in (preferred_categorical or []) if c in df.columns and c not in drop_cols]

    numeric_cols = numeric_candidates + [c for c in df.select_dtypes(include='number').columns if c not in drop_cols and c not in numeric_candidates]
    categorical_cols = categorical_candidates + [c for c in df.select_dtypes(include=['object', 'category', 'bool']).columns if c not in drop_cols and c not in categorical_candidates]

    numeric_cols = numeric_cols[:max_numeric]
    categorical_cols = categorical_cols[:max_categorical]
    feature_cols = numeric_cols + categorical_cols
    X = df[feature_cols].copy()
    y = df[label_col].copy()
    return X, y, numeric_cols, categorical_cols

def make_preprocessor(numeric_cols, categorical_cols, scale=True):
    numeric_steps = [('imputer', SimpleImputer(strategy='median'))]
    if scale:
        numeric_steps.append(('scaler', StandardScaler()))
    categorical_steps = [
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
    return ColumnTransformer(
        transformers=[
            ('num', Pipeline(numeric_steps), numeric_cols),
            ('cat', Pipeline(categorical_steps), categorical_cols)
        ],
        remainder='drop'
    )

## 1. Dataset Snapshot

Start visual analysis by confirming dataset size, label columns, and the fields available for plotting.

In [ ]:
print('PhiUSIIL shape:', phish.shape)
display(phish.head())
if unsw is not None:
    print('UNSW-NB15 shape:', unsw.shape)
    display(unsw.head())

## 2. Class Distribution

Class distribution is usually the first plot in a cybersecurity ML workflow. It tells us whether accuracy will be reliable or misleading.

In [ ]:
phish_plot, unsw_plot = add_target_names(phish, unsw)
fig, axes = plt.subplots(1, 2 if unsw_plot is not None else 1, figsize=(12, 4))
if unsw_plot is None:
    axes = [axes]
sns.countplot(data=phish_plot, x='target_name', ax=axes[0])
axes[0].set_title('PhiUSIIL: URL Label Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Number of Records')
if unsw_plot is not None:
    sns.countplot(data=unsw_plot, x='target_name', ax=axes[1])
    axes[1].set_title('UNSW-NB15: Traffic Label Distribution')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Number of Records')
plt.tight_layout()
plt.show()

print('PhiUSIIL class percentages:')
display((phish_plot['target_name'].value_counts(normalize=True) * 100).round(2).to_frame('percent'))
if unsw_plot is not None:
    print('UNSW-NB15 class percentages:')
    display((unsw_plot['target_name'].value_counts(normalize=True) * 100).round(2).to_frame('percent'))

In [ ]:
if PLOTLY_AVAILABLE:
    display(px.histogram(phish_plot, x='target_name', color='target_name', title='PhiUSIIL Class Distribution'))
    if unsw_plot is not None:
        display(px.histogram(unsw_plot, x='target_name', color='target_name', title='UNSW-NB15 Class Distribution'))
else:
    print('Plotly is not installed. Install it with: pip install plotly')

## 3. Numeric Feature Distributions

Histograms and KDE plots help reveal skew, long tails, and class overlap.

In [ ]:
phish_feature = 'URLLength' if 'URLLength' in phish_plot.columns else phish_plot.select_dtypes(include='number').columns[0]
fig, axes = plt.subplots(1, 2 if unsw_plot is not None else 1, figsize=(13, 4))
if unsw_plot is None:
    axes = [axes]

sns.histplot(data=phish_plot, x=phish_feature, hue='target_name', kde=True, bins=40, ax=axes[0])
axes[0].set_title(f'PhiUSIIL: {phish_feature} by Class')
axes[0].set_xlabel(phish_feature)

if unsw_plot is not None:
    unsw_feature = 'sbytes' if 'sbytes' in unsw_plot.columns else unsw_plot.select_dtypes(include='number').columns[0]
    unsw_plot['log_' + unsw_feature] = np.log1p(unsw_plot[unsw_feature].clip(lower=0))
    sns.histplot(data=unsw_plot, x='log_' + unsw_feature, hue='target_name', kde=True, bins=40, ax=axes[1])
    axes[1].set_title(f'UNSW-NB15: log(1 + {unsw_feature}) by Class')
    axes[1].set_xlabel(f'log(1 + {unsw_feature})')
plt.tight_layout()
plt.show()

## 4. Raw Scale vs. Log Scale

Raw-scale plots can hide most records when a feature has a long right tail. Log scale preserves ordering while making the main body of the distribution visible.

In [ ]:
if unsw_plot is not None and 'sbytes' in unsw_plot.columns:
    feature = 'sbytes'
    temp = sample_df(unsw_plot, n=30000)
    temp['log_value'] = np.log1p(temp[feature].clip(lower=0))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(data=temp, x=feature, bins=50, ax=axes[0])
    axes[0].set_title('Raw Source Bytes')
    axes[0].set_xlabel(feature)
    sns.histplot(data=temp, x='log_value', bins=50, ax=axes[1])
    axes[1].set_title('Log-Transformed Source Bytes')
    axes[1].set_xlabel(f'log(1 + {feature})')
    plt.tight_layout()
    plt.show()
else:
    print('UNSW-NB15 with sbytes is needed for this demonstration.')

## 5. Boxplots by Label

Boxplots compare medians, spread, and outliers between benign and malicious groups.

In [ ]:
fig, axes = plt.subplots(1, 2 if unsw_plot is not None else 1, figsize=(12, 4))
if unsw_plot is None:
    axes = [axes]

sns.boxplot(data=sample_df(phish_plot, 20000), x='target_name', y=phish_feature, ax=axes[0])
axes[0].set_title(f'PhiUSIIL: {phish_feature} by Class')
axes[0].set_xlabel('Class')
axes[0].set_ylabel(phish_feature)

if unsw_plot is not None:
    unsw_y = 'log_sbytes' if 'log_sbytes' in unsw_plot.columns else unsw_plot.select_dtypes(include='number').columns[0]
    sns.boxplot(data=sample_df(unsw_plot, 20000), x='target_name', y=unsw_y, ax=axes[1])
    axes[1].set_title(f'UNSW-NB15: {unsw_y} by Class')
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel(unsw_y)
plt.tight_layout()
plt.show()

## 6. Categorical Count Plots

For categorical security fields, use count plots and top-k filtering so rare categories do not make the chart unreadable.

In [ ]:
fig, axes = plt.subplots(1, 2 if unsw_plot is not None else 1, figsize=(12, 4))
if unsw_plot is None:
    axes = [axes]

phish_cat = 'IsHTTPS' if 'IsHTTPS' in phish_plot.columns else None
if phish_cat is None:
    phish_cat = phish_plot.select_dtypes(include=['object', 'category', 'bool']).columns[0]
sns.countplot(data=phish_plot, x=phish_cat, hue='target_name', ax=axes[0])
axes[0].set_title(f'PhiUSIIL: {phish_cat} by Class')
axes[0].set_xlabel(phish_cat)

if unsw_plot is not None:
    unsw_cat = 'proto' if 'proto' in unsw_plot.columns else unsw_plot.select_dtypes(include=['object', 'category', 'bool']).columns[0]
    top_values = unsw_plot[unsw_cat].value_counts().head(8).index
    sns.countplot(data=unsw_plot[unsw_plot[unsw_cat].isin(top_values)], x=unsw_cat, hue='target_name', ax=axes[1])
    axes[1].set_title(f'UNSW-NB15: Top {unsw_cat} Values by Class')
    axes[1].set_xlabel(unsw_cat)
    axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 7. Scatter Plots by Label

Scatter plots show whether two features separate classes, form clusters, or overlap heavily.

In [ ]:
phish_x = 'URLLength' if 'URLLength' in phish_plot.columns else phish_feature
phish_y = 'URLSimilarityIndex' if 'URLSimilarityIndex' in phish_plot.columns else ('NoOfLettersInURL' if 'NoOfLettersInURL' in phish_plot.columns else phish_x)
fig, axes = plt.subplots(1, 2 if unsw_plot is not None else 1, figsize=(12, 4))
if unsw_plot is None:
    axes = [axes]

sns.scatterplot(data=sample_df(phish_plot, 5000), x=phish_x, y=phish_y, hue='target_name', alpha=0.35, ax=axes[0])
axes[0].set_title(f'PhiUSIIL: {phish_x} vs. {phish_y}')

if unsw_plot is not None and {'sbytes', 'dbytes'}.issubset(unsw_plot.columns):
    temp = sample_df(unsw_plot, 5000).copy()
    temp['log_sbytes'] = np.log1p(temp['sbytes'].clip(lower=0))
    temp['log_dbytes'] = np.log1p(temp['dbytes'].clip(lower=0))
    sns.scatterplot(data=temp, x='log_sbytes', y='log_dbytes', hue='target_name', alpha=0.35, ax=axes[1])
    axes[1].set_title('UNSW-NB15: Source vs. Destination Bytes')
plt.tight_layout()
plt.show()

In [ ]:
if PLOTLY_AVAILABLE:
    display(px.scatter(sample_df(phish_plot, 4000), x=phish_x, y=phish_y, color='target_name',
                       title=f'Interactive PhiUSIIL Scatter: {phish_x} vs. {phish_y}'))
    if unsw_plot is not None and {'sbytes', 'dbytes'}.issubset(unsw_plot.columns):
        temp = sample_df(unsw_plot, 4000).copy()
        temp['log_sbytes'] = np.log1p(temp['sbytes'].clip(lower=0))
        temp['log_dbytes'] = np.log1p(temp['dbytes'].clip(lower=0))
        display(px.scatter(temp, x='log_sbytes', y='log_dbytes', color='target_name',
                           title='Interactive UNSW-NB15 Scatter: Source vs. Destination Bytes'))

## 8. Correlation Heatmaps

Correlation heatmaps reveal redundancy, feature groups, and possible leakage signals. Correlation with the label should trigger investigation, not automatic trust.

In [ ]:
phish_preferred = ['URLLength', 'DomainLength', 'NoOfSubDomain', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfOtherSpecialCharsInURL', 'URLSimilarityIndex', 'IsHTTPS']
phish_num, phish_num_cols = safe_numeric_frame(phish_plot, phish_preferred, max_cols=8)

fig, axes = plt.subplots(1, 2 if unsw_plot is not None else 1, figsize=(14, 5))
if unsw_plot is None:
    axes = [axes]
sns.heatmap(phish_num.corr(), cmap='coolwarm', center=0, ax=axes[0])
axes[0].set_title('PhiUSIIL: Numeric Feature Correlations')

if unsw_plot is not None:
    unsw_preferred = ['dur', 'sbytes', 'dbytes', 'spkts', 'dpkts', 'rate', 'sttl', 'dttl']
    unsw_num, unsw_num_cols = safe_numeric_frame(unsw_plot, unsw_preferred, max_cols=8)
    sns.heatmap(unsw_num.corr(), cmap='coolwarm', center=0, ax=axes[1])
    axes[1].set_title('UNSW-NB15: Numeric Feature Correlations')
plt.tight_layout()
plt.show()

## 9. From Chart to Security Decision

A chart should produce a follow-up question. For example, a rare attack class points to precision and recall checks; a highly skewed byte feature suggests log transformation; a near-perfect target correlation requires leakage review.